# Infrastructure Atlas — Southeast Asia

Visualizing infrastructure across **Thailand, Vietnam, Indonesia & Malaysia** on interactive maps.

### Data Sources
- **World Bank Indicators API** — economic & infrastructure metrics (open access)
- **Natural Earth** — country boundary geometries (110m resolution)
- **OpenStreetMap / Overpass API** — roads, railways, industrial & agricultural land use

### Infrastructure Categories
| Category | World Bank Indicators | OSM Tags |
|----------|----------------------|----------|
| Roads | `IS.ROD.TOTL.KM` (total network km) | `highway=*` |
| Rail | `IS.RRS.TOTL.KM` (rail lines km) | `railway=rail` |
| Manufacturing | `NV.IND.MANF.ZS` (% of GDP) | `landuse=industrial` |
| Mining | `NY.GDP.MINR.RT.ZS` (mineral rents % GDP) | `landuse=quarry` |
| Agriculture | `AG.LND.AGRI.ZS` (% of land area) | `landuse=farmland` |
| Energy | `EG.ELC.ACCS.ZS` (electricity access %) | — |
| Resources | `NY.GDP.TOTL.RT.ZS` (total resource rents) | — |

In [1]:
# ── Cell 1: Fetch World Bank infrastructure indicators ──────────────────────
import requests
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Countries: Thailand, Vietnam, Indonesia, Malaysia
COUNTRIES = {"THA": "Thailand", "VNM": "Vietnam", "IDN": "Indonesia", "MYS": "Malaysia"}
COUNTRY_CODES = ";".join(COUNTRIES.keys())

# World Bank indicator codes for infrastructure
INDICATORS = {
    "IS.ROD.TOTL.KM":   "Roads — total network (km)",
    "IS.RRS.TOTL.KM":   "Rail lines (total route-km)",
    "NV.IND.MANF.ZS":   "Manufacturing, value added (% of GDP)",
    "NV.IND.TOTL.ZS":   "Industry, value added (% of GDP)",
    "NY.GDP.MINR.RT.ZS": "Mineral rents (% of GDP)",
    "NY.GDP.TOTL.RT.ZS": "Total natural resources rents (% of GDP)",
    "AG.LND.AGRI.ZS":   "Agricultural land (% of land area)",
    "AG.LND.ARBL.ZS":   "Arable land (% of land area)",
    "EG.ELC.ACCS.ZS":   "Access to electricity (% of population)",
    "EG.ELC.PROD.KH":   "Electricity production (kWh)",
    "NY.GDP.MKTP.CD":   "GDP (current US$)",
    "NY.GDP.PCAP.CD":   "GDP per capita (current US$)",
    "SP.POP.TOTL":      "Population, total",
}

def fetch_wb(indicator, countries=COUNTRY_CODES, date="2000:2023"):
    """Fetch a single indicator from the World Bank API."""
    url = f"https://api.worldbank.org/v2/country/{countries}/indicator/{indicator}"
    all_rows = []
    page = 1
    while True:
        params = {"format": "json", "per_page": 500, "date": date, "page": page}
        r = requests.get(url, params=params, timeout=30)
        raw = r.json()
        if len(raw) < 2 or raw[1] is None:
            break
        for rec in raw[1]:
            if rec["value"] is not None:
                all_rows.append({
                    "country": rec["country"]["value"],
                    "iso3": rec["countryiso3code"],
                    "year": int(rec["date"]),
                    "value": rec["value"],
                })
        if page >= raw[0]["pages"]:
            break
        page += 1
    return all_rows

print("Fetching World Bank indicators …")
all_data = {}
for code, label in INDICATORS.items():
    rows = fetch_wb(code)
    all_data[code] = pd.DataFrame(rows)
    n = len(rows)
    status = "✓" if n > 0 else "✗ (no data)"
    print(f"  {status} {code:<25s} → {n:>4d} rows  │ {label}")

print(f"\nAll {len(INDICATORS)} indicators fetched ✓")

Fetching World Bank indicators …
  ✗ (no data) IS.ROD.TOTL.KM            →    0 rows  │ Roads — total network (km)
  ✓ IS.RRS.TOTL.KM            →   59 rows  │ Rail lines (total route-km)
  ✓ NV.IND.MANF.ZS            →   91 rows  │ Manufacturing, value added (% of GDP)
  ✓ NV.IND.TOTL.ZS            →   96 rows  │ Industry, value added (% of GDP)
  ✓ NY.GDP.MINR.RT.ZS         →   88 rows  │ Mineral rents (% of GDP)
  ✓ NY.GDP.TOTL.RT.ZS         →   88 rows  │ Total natural resources rents (% of GDP)
  ✓ AG.LND.AGRI.ZS            →   96 rows  │ Agricultural land (% of land area)
  ✓ AG.LND.ARBL.ZS            →   96 rows  │ Arable land (% of land area)
  ✓ EG.ELC.ACCS.ZS            →   96 rows  │ Access to electricity (% of population)
  ✗ (no data) EG.ELC.PROD.KH            →    0 rows  │ Electricity production (kWh)
  ✓ NY.GDP.MKTP.CD            →   96 rows  │ GDP (current US$)
  ✓ NY.GDP.PCAP.CD            →   96 rows  │ GDP per capita (current US$)
  ✓ SP.POP.TOTL               →   9

In [2]:
# ── Cell 2: Latest snapshot — summary table ──────────────────────────────────

def latest_value(df, iso3):
    """Get the most recent non-null value for a country."""
    if df.empty or "iso3" not in df.columns:
        return None, None
    sub = df[df["iso3"] == iso3].sort_values("year", ascending=False)
    if sub.empty:
        return None, None
    row = sub.iloc[0]
    return row["value"], int(row["year"])

summary_rows = []
for iso3, name in COUNTRIES.items():
    row = {"Country": name, "ISO3": iso3}
    for code, label in INDICATORS.items():
        df = all_data[code]
        val, yr = latest_value(df, iso3)
        if val is not None:
            row[f"{label} ({yr})"] = val
        else:
            row[label] = None
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index("Country")
print("Latest available data per country:\n")
df_summary.T

Latest available data per country:



Country,Thailand,Vietnam,Indonesia,Malaysia
ISO3,THA,VNM,IDN,MYS
Roads — total network (km),None,None,None,None
Rail lines (total route-km) (2017),4092.0,NaN,NaN,NaN
"Manufacturing, value added (% of GDP) (2023)",24.979281,24.179672,18.667416,23.020234
"Industry, value added (% of GDP) (2023)",32.980409,37.58157,40.215682,37.564857
Mineral rents (% of GDP) (2021),0.0,0.102201,1.90958,0.0
Total natural resources rents (% of GDP) (2021),1.817027,2.547017,5.15704,6.921966
Agricultural land (% of land area) (2023),43.798078,39.214738,29.125806,26.087354
Arable land (% of land area) (2023),30.995322,21.467318,9.396239,2.387156
Access to electricity (% of population) (2023),100.0,99.8,99.4,100.0


In [3]:
# ── Cell 3: Load country boundaries (Natural Earth) ──────────────────────────
import geopandas as gpd

NE_URL = "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip"
print("Downloading Natural Earth 50m boundaries …")
world = gpd.read_file(NE_URL)

sea_names = list(COUNTRIES.values())
gdf = world[world["ADMIN"].isin(sea_names)][["ADMIN", "ISO_A3", "geometry"]].copy()
gdf = gdf.rename(columns={"ADMIN": "country", "ISO_A3": "iso3"})
print(f"Loaded {len(gdf)} country boundaries: {', '.join(gdf['country'].tolist())}")
gdf.head()

Loaded 4 country boundaries: Vietnam, Thailand, Malaysia, Indonesia


,country,iso3,geometry
3,Vietnam,VNM,"MULTIPOLYGON (((104.06396 10.39082, 104.08301 ..."
42,Thailand,THA,"MULTIPOLYGON (((102.60645 11.67651, 102.58994 ..."
115,Malaysia,MYS,"MULTIPOLYGON (((100.11914 6.44199, 100.13799 6..."
143,Indonesia,IDN,"MULTIPOLYGON (((97.48154 1.46509, 97.69834 1.1..."


In [4]:
# ── Cell 4: Fetch OSM infrastructure via Overpass API ────────────────────────
# We query major roads, railways, and industrial/mining/farm land use
# Using centroids + small bounding boxes to keep queries fast

import time

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Bounding boxes for each country (approx: south, west, north, east)
BBOXES = {
    "Thailand":  (5.6, 97.3, 20.5, 105.7),
    "Vietnam":   (8.2, 102.1, 23.4, 109.5),
    "Indonesia": (-11.0, 95.0, 6.1, 141.0),
    "Malaysia":  (0.8, 99.6, 7.4, 119.3),
}

def overpass_query(query_str):
    """Run an Overpass API query and return the JSON response."""
    r = requests.post(OVERPASS_URL, data={"data": query_str}, timeout=120)
    r.raise_for_status()
    return r.json()

def count_features(country, bbox, tag_filter, label):
    """Count OSM features matching a tag filter in a bbox."""
    s, w, n, e = bbox
    query = f"""
    [out:json][timeout:60];
    (
      nwr[{tag_filter}]({s},{w},{n},{e});
    );
    out count;
    """
    try:
        data = overpass_query(query)
        count = int(data["elements"][0]["tags"]["total"])
        return count
    except Exception as ex:
        print(f"    ⚠ {country}/{label}: {ex}")
        return None

# Tags to query
OSM_TAGS = {
    "Major roads":    '"highway"~"motorway|trunk|primary"',
    "Railways":       '"railway"="rail"',
    "Industrial":     '"landuse"="industrial"',
    "Mining/quarry":  '"landuse"="quarry"',
    "Farmland":       '"landuse"="farmland"',
}

print("Querying OpenStreetMap Overpass API (this may take a minute) …\n")
osm_rows = []
for country, bbox in BBOXES.items():
    print(f"  {country}:")
    row = {"country": country}
    for label, tag in OSM_TAGS.items():
        count = count_features(country, bbox, tag, label)
        row[label] = count
        print(f"    {label:<18s} → {count:>8,}" if count else f"    {label:<18s} → —")
        time.sleep(2)  # be polite to the API
    osm_rows.append(row)
    print()

df_osm = pd.DataFrame(osm_rows).set_index("country")
print("OSM feature counts:")
df_osm

Querying OpenStreetMap Overpass API (this may take a minute) …

  Thailand:
    ⚠ Thailand/Major roads: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter
    Major roads        → —
    ⚠ Thailand/Railways: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter
    Railways           → —
    ⚠ Thailand/Industrial: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter
    Industrial         → —
    ⚠ Thailand/Mining/quarry: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter
    Mining/quarry      → —
    ⚠ Thailand/Farmland: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter
    Farmland           → —

  Vietnam:
    ⚠ Vietnam/Major roads: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter
    Major roads        → —
    ⚠ Vietnam/Railways: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/in

,Major roads,Railways,Industrial,Mining/quarry,Farmland
country,,,,,
Thailand,None,None,None,None,None
Vietnam,None,None,None,None,None
Indonesia,None,None,None,None,None
Malaysia,None,None,None,None,None


In [5]:
# ── Cell 5: Merge all data & build infrastructure score ──────────────────────

# Combine WB latest values + OSM counts into a single DataFrame
infra = pd.DataFrame(list(COUNTRIES.items()), columns=["iso3", "country"])

# Add latest WB values
wb_cols = {
    "IS.ROD.TOTL.KM":   "roads_km",
    "IS.RRS.TOTL.KM":   "rail_km",
    "NV.IND.MANF.ZS":   "manufacturing_pct_gdp",
    "NY.GDP.MINR.RT.ZS": "mineral_rents_pct_gdp",
    "AG.LND.AGRI.ZS":   "agri_land_pct",
    "EG.ELC.ACCS.ZS":   "electricity_access_pct",
    "NY.GDP.MKTP.CD":   "gdp_usd",
    "NY.GDP.PCAP.CD":   "gdp_per_capita",
    "SP.POP.TOTL":      "population",
}

for code, col_name in wb_cols.items():
    df = all_data[code]
    vals = []
    for iso3 in infra["iso3"]:
        v, _ = latest_value(df, iso3)
        vals.append(v)
    infra[col_name] = vals

# Merge OSM counts
infra = infra.merge(df_osm.reset_index(), on="country", how="left")

print("Combined infrastructure dataset:")
infra.set_index("country").T

Combined infrastructure dataset:


country,Thailand,Vietnam,Indonesia,Malaysia
iso3,THA,VNM,IDN,MYS
roads_km,None,None,None,None
rail_km,4092.0,3159.0,5483.0,1655.0
manufacturing_pct_gdp,24.979281,24.179672,18.667416,23.020234
mineral_rents_pct_gdp,0.0,0.102201,1.90958,0.0
agri_land_pct,43.798078,39.214738,29.125806,26.087354
electricity_access_pct,100.0,99.8,99.4,100.0
gdp_usd,515906283940.932983,433857681378.291016,1371169301563.620117,399949418752.656982
gdp_per_capita,7195.101309,4323.35032,4876.307745,11386.039564
population,71702435,100352192,281190067,35126298


In [6]:
# ── Cell 6: Time-series chart — infrastructure indicators over time ─────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Color palette — muted earthy tones consistent with other notebooks
PALETTE = {
    "Thailand":  "#4F7F84",  # teal
    "Vietnam":   "#C4787A",  # dusty rose
    "Indonesia": "#E9C46A",  # gold
    "Malaysia":  "#2A9D8F",  # green
}
BG_COLOR = "#f5efe2"
TEXT_COLOR = "#2f2a25"
GRID_COLOR = "#ddd5c4"
FONT = "Georgia"

# Indicators to plot as 2×2 subplots
PLOT_INDICATORS = [
    ("NV.IND.MANF.ZS",   "Manufacturing (% of GDP)",        ".1f", "%"),
    ("AG.LND.AGRI.ZS",   "Agricultural Land (% of area)",   ".1f", "%"),
    ("NY.GDP.TOTL.RT.ZS", "Natural Resource Rents (% GDP)",  ".1f", "%"),
    ("EG.ELC.ACCS.ZS",   "Electricity Access (%)",          ".1f", "%"),
]

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.12, horizontal_spacing=0.10)

for idx, (code, title, fmt, suffix) in enumerate(PLOT_INDICATORS):
    row, col = divmod(idx, 2)
    row += 1; col += 1
    df = all_data[code]
    for iso3, name in COUNTRIES.items():
        sub = df[df["iso3"] == iso3].sort_values("year")
        fig.add_trace(go.Scatter(
            x=sub["year"], y=sub["value"],
            mode="lines+markers",
            name=name,
            legendgroup=name,
            showlegend=(idx == 0),
            line=dict(color=PALETTE[name], width=2),
            marker=dict(size=4),
            hovertemplate=f"<b>{name}</b><br>%{{x}}: %{{y:{fmt}}}{suffix}<extra></extra>",
        ), row=row, col=col)

# Layout
subplot_titles = [t for _, t, _, _ in PLOT_INDICATORS]
for i, title in enumerate(subplot_titles):
    r, c = divmod(i, 2)
    x_pos = 0.225 if c == 0 else 0.775
    y_pos = 1.0 if r == 0 else 0.44
    fig.add_annotation(
        text=title, x=x_pos, y=y_pos,
        xref="paper", yref="paper", xanchor="center", yanchor="bottom",
        showarrow=False, font=dict(family=FONT, size=14, color=TEXT_COLOR),
    )

fig.update_layout(
    height=850, width=1100,
    paper_bgcolor=BG_COLOR, plot_bgcolor=BG_COLOR,
    font=dict(family=FONT, color=TEXT_COLOR),
    hovermode="x unified",
    legend=dict(
        x=1.02, y=1, xanchor="left", yanchor="top",
        bgcolor=f"rgba(245,239,226,0.85)", font=dict(size=11),
    ),
    margin=dict(l=70, r=170, t=100, b=50),
    title=dict(
        text="Infrastructure Indicators — Southeast Asia (2000–2023)",
        font=dict(family=FONT, size=20, color=TEXT_COLOR),
        x=0.5, xanchor="center",
    ),
)
fig.update_xaxes(gridcolor=GRID_COLOR, linecolor=GRID_COLOR)
fig.update_yaxes(gridcolor=GRID_COLOR, linecolor=GRID_COLOR)

fig.write_html("output/infrastructure_timeseries.html")
print("Saved → output/infrastructure_timeseries.html")
fig.show()

Saved → output/infrastructure_timeseries.html


In [7]:
# ── Cell 7: PyDeck 3D choropleth map — infrastructure by country ─────────────
import pydeck as pdk
import numpy as np

# Merge infrastructure data with geometries
map_gdf = gdf.merge(infra, on="country", how="left", suffixes=("", "_infra"))

# Convert to GeoJSON-like format for PyDeck
map_gdf["manufacturing_pct_gdp"] = map_gdf["manufacturing_pct_gdp"].fillna(0)
map_gdf["gdp_per_capita"] = map_gdf["gdp_per_capita"].fillna(0)

# Use manufacturing % of GDP to drive extrusion height
map_gdf["elevation"] = map_gdf["manufacturing_pct_gdp"] * 8000

# Color by GDP per capita — map to a warm palette
def gdp_to_color(gdp_pc):
    """Map GDP per capita to an RGBA color."""
    max_gdp = 15000
    t = min(gdp_pc / max_gdp, 1.0)
    # Warm gradient: dark brown → amber → cream
    r = int(79 + t * (233 - 79))
    g = int(55 + t * (196 - 55))
    b = int(45 + t * (106 - 45))
    return [r, g, b, 200]

map_gdf["fill_color"] = map_gdf["gdp_per_capita"].apply(gdp_to_color)

# Build GeoJSON features
features = []
for _, row in map_gdf.iterrows():
    geom = row["geometry"].__geo_interface__
    props = {
        "country": row["country"],
        "elevation": row["elevation"],
        "fill_color": row["fill_color"],
        "manufacturing_pct": round(row["manufacturing_pct_gdp"], 1),
        "gdp_per_capita": round(row["gdp_per_capita"], 0),
        "population": int(row["population"]) if pd.notna(row["population"]) else 0,
        "agri_land_pct": round(row["agri_land_pct"], 1) if pd.notna(row["agri_land_pct"]) else 0,
        "electricity_pct": round(row["electricity_access_pct"], 1) if pd.notna(row["electricity_access_pct"]) else 0,
    }
    features.append({"type": "Feature", "geometry": geom, "properties": props})

geojson = {"type": "FeatureCollection", "features": features}

# PyDeck layer
layer = pdk.Layer(
    "GeoJsonLayer",
    data=geojson,
    get_elevation="properties.elevation",
    elevation_scale=1,
    extruded=True,
    wireframe=True,
    get_fill_color="properties.fill_color",
    get_line_color=[60, 50, 40, 120],
    line_width_min_pixels=1,
    pickable=True,
    auto_highlight=True,
)

view = pdk.ViewState(
    latitude=5.0, longitude=110.0,
    zoom=3.8, pitch=45, bearing=-15,
)

tooltip = {
    "html": """
    <div style='font-family:Georgia; padding:8px; background:#2f2a25; color:#f5efe2; border-radius:6px;'>
        <b style='font-size:14px;'>{country}</b><br/>
        <hr style='border-color:#7f7a75; margin:4px 0;'/>
        Manufacturing: {manufacturing_pct}% of GDP<br/>
        GDP per capita: ${gdp_per_capita}<br/>
        Population: {population}<br/>
        Agricultural land: {agri_land_pct}%<br/>
        Electricity access: {electricity_pct}%
    </div>
    """,
    "style": {"backgroundColor": "transparent", "color": "white"},
}

# Use CARTO dark basemap (free, no API key needed)
deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

deck.to_html("output/infrastructure_map.html")
print("Saved → output/infrastructure_map.html")
deck

Saved → output/infrastructure_map.html


{
  "initialViewState": {
    "bearing": -15,
    "latitude": 5.0,
    "longitude": 110.0,
    "pitch": 45,
    "zoom": 3.8
  },
  "layers": [
    {
      "@@type": "GeoJsonLayer",
      "autoHighlight": true,
      "data": {
        "features": [
          {
            "geometry": {
              "coordinates": [
                [
                  [
                    [
                      104.06396484375,
                      10.390820312499997
                    ],
                    [
                      104.0830078125,
                      10.341113281249989
                    ],
                    [
                      104.07578125000003,
                      10.224853515625
                    ],
                    [
                      104.03681640625001,
                      10.110742187499994
                    ],
                    [
                      104.04833984375,
                      10.06103515625
                    ],
                    [
                      104.01845703125002,
                      10.029199218749994
                    ],
                    [
                      103.9521484375,
                      10.242919921875
                    ],
                    [
                      103.86796874999999,
                      10.335400390624997
                    ],
                    [
                      103.84951171875002,
                      10.37109375
                    ],
                    [
                      103.8984375,
                      10.368505859374991
                    ],
                    [
                      103.98583984375,
                      10.426953124999997
                    ],
                    [
                      104.02773437500002,
                      10.428369140624994
                    ],
                    [
                      104.06396484375,
                      10.390820312499997
                    ]
                  ]
                ],
                [
                  [
                    [
                      107.97265625,
                      21.507958984374994
                    ],
                    [
                      107.92578125,
                      21.498925781249994
                    ],
                    [
                      107.80908203125,
                      21.497119140625003
                    ],
                    [
                      107.70722656250001,
                      21.40585937499999
                    ],
                    [
                      107.63671875,
                      21.368066406249994
                    ],
                    [
                      107.52695312500003,
                      21.336230468750003
                    ],
                    [
                      107.40996093749999,
                      21.284814453124994
                    ],
                    [
                      107.37617187500001,
                      21.194140625000003
                    ],
                    [
                      107.37333984374999,
                      21.128466796875003
                    ],
                    [
                      107.35429687499999,
                      21.05517578125
                    ],
                    [
                      107.16474609375001,
                      20.94873046875
                    ],
                    [
                      107.11171875000002,
                      20.95957031249999
                    ],
                    [
                      107.0751953125,
                      20.999267578125
                    ],
                    [
                      107.01923828125001,
                      20.9912109375
                    ],
                    [
                      106.9814453125,
                      20.971386718749997
                    ],
                    [
                      106.93642578125002,
            